## Full Year DF Prep

In [1]:
# confirm all files
from pathlib import Path

DATA_DIR = Path.home() / "datasets" / "nyc-taxi" / "2025"

files = sorted(DATA_DIR.glob("yellow_tripdata_2025-??.parquet"))

print("Files found:", len(files))

for f in files:
    print(f.name, f"{f.stat().st_size / 1024**2:.1f} MB")

Files found: 12
yellow_tripdata_2025-01.parquet 56.4 MB
yellow_tripdata_2025-02.parquet 57.5 MB
yellow_tripdata_2025-03.parquet 66.7 MB
yellow_tripdata_2025-04.parquet 64.2 MB
yellow_tripdata_2025-05.parquet 74.2 MB
yellow_tripdata_2025-06.parquet 70.1 MB
yellow_tripdata_2025-07.parquet 63.8 MB
yellow_tripdata_2025-08.parquet 59.4 MB
yellow_tripdata_2025-09.parquet 69.1 MB
yellow_tripdata_2025-10.parquet 71.8 MB
yellow_tripdata_2025-11.parquet 67.8 MB
yellow_tripdata_2025-12.parquet 70.3 MB


In [2]:
# compressed size
total_size_gb = sum(f.stat().st_size for f in files) / 1024**3

print(f"Total Parquet size: {total_size_gb:.2f} GB")

Total Parquet size: 0.77 GB


In [3]:
# compare schemas
import pandas as pd
import pyarrow.parquet as pq

for f in files:
    table = pq.read_schema(f)
    print(f.name, len(table.names))

yellow_tripdata_2025-01.parquet 20
yellow_tripdata_2025-02.parquet 20
yellow_tripdata_2025-03.parquet 20
yellow_tripdata_2025-04.parquet 20
yellow_tripdata_2025-05.parquet 20
yellow_tripdata_2025-06.parquet 20
yellow_tripdata_2025-07.parquet 20
yellow_tripdata_2025-08.parquet 20
yellow_tripdata_2025-09.parquet 20
yellow_tripdata_2025-10.parquet 20
yellow_tripdata_2025-11.parquet 20
yellow_tripdata_2025-12.parquet 20


In [4]:
schemas = {
    f.name: pq.read_schema(f).names
    for f in files
}

first_columns = schemas[files[0].name]

for name, columns in schemas.items():
    print(name, columns == first_columns)

yellow_tripdata_2025-01.parquet True
yellow_tripdata_2025-02.parquet True
yellow_tripdata_2025-03.parquet True
yellow_tripdata_2025-04.parquet True
yellow_tripdata_2025-05.parquet True
yellow_tripdata_2025-06.parquet True
yellow_tripdata_2025-07.parquet True
yellow_tripdata_2025-08.parquet True
yellow_tripdata_2025-09.parquet True
yellow_tripdata_2025-10.parquet True
yellow_tripdata_2025-11.parquet True
yellow_tripdata_2025-12.parquet True


In [5]:
# row counts without loading into memory
row_counts = {}

for f in files:
    metadata = pq.ParquetFile(f).metadata
    row_counts[f.name] = metadata.num_rows

for name, rows in row_counts.items():
    print(f"{name}: {rows:,}")

yellow_tripdata_2025-01.parquet: 3,475,226
yellow_tripdata_2025-02.parquet: 3,577,543
yellow_tripdata_2025-03.parquet: 4,145,257
yellow_tripdata_2025-04.parquet: 3,970,553
yellow_tripdata_2025-05.parquet: 4,591,845
yellow_tripdata_2025-06.parquet: 4,322,960
yellow_tripdata_2025-07.parquet: 3,898,963
yellow_tripdata_2025-08.parquet: 3,574,091
yellow_tripdata_2025-09.parquet: 4,251,015
yellow_tripdata_2025-10.parquet: 4,428,699
yellow_tripdata_2025-11.parquet: 4,181,444
yellow_tripdata_2025-12.parquet: 4,305,006


In [6]:
total_rows = sum(row_counts.values())

print(f"Total 2025 rows: {total_rows:,}")

Total 2025 rows: 48,722,602


In [7]:
# choice not to concatenate all files into a single DataFrame due to memory constraints. Instead, we will process each file individually and aggregate results as needed.

In [2]:
# cleaning all files, removing out of bounds or otherwise nonsensical data
from pathlib import Path

DATA_DIR = Path.home() / "datasets" / "nyc-taxi" / "2025"
CLEAN_DIR = DATA_DIR / "cleaned"
CLEAN_DIR.mkdir(exist_ok=True)

In [4]:
import pandas as pd

summary = []

for f in files:
    print(f"Processing {f.name}...")

    df = pd.read_parquet(f)

    original_rows = len(df)

    duration_minutes = (
        df["tpep_dropoff_datetime"]
        - df["tpep_pickup_datetime"]
    ).dt.total_seconds() / 60

    valid_mask = (
        (duration_minutes >= 0)
        & (duration_minutes <= 1440)
    )

    df_clean = df.loc[valid_mask].copy()

    removed_rows = original_rows - len(df_clean)

    output_path = CLEAN_DIR / f.name.replace(
        ".parquet",
        "_clean.parquet"
    )

    df_clean.to_parquet(output_path, index=False)

    summary.append({
        "file": f.name,
        "original_rows": original_rows,
        "clean_rows": len(df_clean),
        "removed_rows": removed_rows,
    })

    del df
    del df_clean

Processing yellow_tripdata_2025-01.parquet...
Processing yellow_tripdata_2025-02.parquet...
Processing yellow_tripdata_2025-03.parquet...
Processing yellow_tripdata_2025-04.parquet...
Processing yellow_tripdata_2025-05.parquet...
Processing yellow_tripdata_2025-06.parquet...
Processing yellow_tripdata_2025-07.parquet...
Processing yellow_tripdata_2025-08.parquet...
Processing yellow_tripdata_2025-09.parquet...
Processing yellow_tripdata_2025-10.parquet...
Processing yellow_tripdata_2025-11.parquet...
Processing yellow_tripdata_2025-12.parquet...


In [5]:
summary_df = pd.DataFrame(summary)

summary_df

,file,original_rows,clean_rows,removed_rows
0,yellow_tripdata_2025-01.parquet,3475226,3475082,144
1,yellow_tripdata_2025-02.parquet,3577543,3577434,109
2,yellow_tripdata_2025-03.parquet,4145257,4145165,92
3,yellow_tripdata_2025-04.parquet,3970553,3970365,188
4,yellow_tripdata_2025-05.parquet,4591845,4591718,127
5,yellow_tripdata_2025-06.parquet,4322960,4322709,251
6,yellow_tripdata_2025-07.parquet,3898963,3898931,32
7,yellow_tripdata_2025-08.parquet,3574091,3574044,47
8,yellow_tripdata_2025-09.parquet,4251015,4250978,37
9,yellow_tripdata_2025-10.parquet,4428699,4428662,37


In [6]:
# cleaning analysis
print(
    "Original rows:",
    f"{summary_df['original_rows'].sum():,}"
)

print(
    "Clean rows:",
    f"{summary_df['clean_rows'].sum():,}"
)

print(
    "Removed rows:",
    f"{summary_df['removed_rows'].sum():,}"
)

Original rows: 48,722,602
Clean rows: 48,720,015
Removed rows: 2,587


In [12]:
# save summary to CSV
summary_df.to_csv(
    CLEAN_DIR / "cleaning_summary.csv",
    index=False
)